# Sesion 01 - Lab 2: Creacion y comparacion de compute

Este laboratorio crea un job cluster y un SQL warehouse serverless desde codigo, mide sus tiempos de arranque y compara la latencia de una misma consulta ejecutada en cada uno. Antes de correrlo, sube `ventas_demo.csv` al volume `/Volumes/dbassociate/default/vol_landing/` (arrastrando el archivo en el explorador de Catalog, o con `dbutils.fs.cp` si ya esta en el workspace).

## Verificacion del entorno

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import CreateWarehouseRequestWarehouseType
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
import time

w = WorkspaceClient()

dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/")

versiones_lts = [v for v in w.clusters.spark_versions().versions if "LTS" in v.name and "ML" not in v.name]
for v in versiones_lts[:5]:
    print(v.key, "-", v.name)


## Lab 2A: Crear un job cluster efimero via SDK

Se usa la primera version LTS disponible en la lista anterior. Nunca una version Beta o de feature development para este tipo de carga.

In [ ]:
runtime_lts = versiones_lts[0].key
print("Runtime LTS elegido:", runtime_lts)

inicio = time.time()

cluster_reto = w.clusters.create(
    cluster_name="sesion01-lab2-job-cluster",
    spark_version=runtime_lts,
    node_type_id="Standard_DS3_v2",
    num_workers=1,
    autotermination_minutes=20,
).result()

segundos_arranque_cluster = time.time() - inicio
print(f"Cluster {cluster_reto.cluster_id} listo en {segundos_arranque_cluster:.1f} segundos")


## Lab 2B: Crear un SQL warehouse serverless via SDK

En la API, un warehouse serverless se configura como tipo `PRO` con `enable_serverless_compute=True`; no existe un tipo `SERVERLESS` separado.

In [ ]:
inicio = time.time()

warehouse_reto = w.warehouses.create(
    name="sesion01-lab2-warehouse",
    cluster_size="2X-Small",
    warehouse_type=CreateWarehouseRequestWarehouseType.PRO,
    enable_serverless_compute=True,
    auto_stop_mins=10,
).result()

segundos_arranque_warehouse = time.time() - inicio
print(f"SQL warehouse {warehouse_reto.id} listo en {segundos_arranque_warehouse:.1f} segundos")


## Lab 2C: Ejecutar la misma consulta en ambos tipos de compute

Se define un `StructType` explicito en vez de `inferSchema=True`, el mismo criterio que se exige desde la capa Bronze en sesiones posteriores.

In [ ]:
ventas_path = "/Volumes/dbassociate/default/vol_landing/sesion_01/ventas_demo.csv"

schema_ventas = StructType([
    StructField("id_venta", IntegerType(), False),
    StructField("fecha_venta", DateType(), False),
    StructField("region", StringType(), True),
    StructField("canal", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("producto", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", DoubleType(), True),
    StructField("monto_total", DoubleType(), True),
])

df_ventas = spark.read.option("header", True).schema(schema_ventas).csv(ventas_path)
df_ventas.write.mode("overwrite").saveAsTable("dbassociate.default.ventas_demo_lab2")
# df_ventas.write.mode("overwrite").save("/Volumes/dbassociate/default/vol_landing/sesion_01/ventas_demo")

print("Filas cargadas:", df_ventas.count())


In [ ]:
consulta = """
    SELECT region, categoria, SUM(monto_total) AS monto_total, COUNT(*) AS num_ventas
    FROM dbassociate.default.ventas_demo_lab2
    GROUP BY region, categoria
    ORDER BY monto_total DESC
"""

inicio = time.time()
resultado_cluster = spark.sql(consulta).collect()
segundos_query_cluster = time.time() - inicio
print(f"Query sobre el cluster actual: {segundos_query_cluster:.2f} s ({len(resultado_cluster)} filas)")


In [ ]:
inicio = time.time()
resultado_warehouse = w.statement_execution.execute_statement(
    warehouse_id=warehouse_reto.id,
    statement=consulta,
    wait_timeout="30s",
)
segundos_query_warehouse = time.time() - inicio

print(f"Query sobre el cluster actual: {segundos_query_cluster:.2f} s")
print(f"Query sobre el SQL warehouse serverless: {segundos_query_warehouse:.2f} s (incluye arranque si el warehouse estaba detenido)")


## Lab 2D: Tabla comparativa de costos y criterios

Los tiempos exactos de arranque y de DBU consumidas dependen de la region, el tier del workspace y si Photon esta activo, asi que no se fijan numeros absolutos: se compara con los tiempos medidos arriba, en este mismo workspace.

| Criterio | Job cluster (classic) | SQL warehouse serverless |
|---|---|---|
| Arranque tipico | minutos (ver `segundos_arranque_cluster`) | segundos (ver `segundos_arranque_warehouse`) |
| Autoscaling | manual, configurado al crear el cluster | administrado por Databricks |
| Concurrencia | limitada por el numero de workers fijado | escala segun demanda |
| Facturacion | por el tiempo total que el cluster esta activo | por el tiempo de computo real de las queries |


## Limpieza

In [ ]:
w.clusters.permanent_delete(cluster_reto.cluster_id)
w.warehouses.delete(warehouse_reto.id)
spark.sql("DROP TABLE IF EXISTS dbassociate.default.ventas_demo_lab2")

print("Cluster, warehouse y tabla temporal de este laboratorio eliminados.")
